<a href="https://colab.research.google.com/github/ahlqui/VeloxChemColabs/blob/main/AtomicCharges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# Calculate partial charges


This notebook calculates partial atomic charges from a SMILES code.

A molecule can be defined using a SMILES code. Two suggested ways to get a SMILES:
1. Sketch your molecule at https://www.rcsb.org/chemical-sketch — the SMILES code appears below the structure.
2. Build your molecule at https://molview.org/, go to Tools / Information Card.

Example SMILES for CO2: `O=C=O`


In [17]:
# Local environment check for QuimicaAutomocio P2.
# The original Colab version used a Colab-only installation cell.
# In local Jupyter, skip that installation cell and use the conda environment instead:
#   conda activate quimica-echem

import sys
try:
    import veloxchem as vlx
    print(f"VeloxChem imported correctly: {getattr(vlx, '__version__', 'unknown')}")
    print(f"Python executable: {sys.executable}")
except Exception as exc:
    raise RuntimeError(
        "VeloxChem is not available in this kernel. Activate the conda environment "
        "with `conda activate quimica-echem`, start Jupyter from that terminal, "
        "and select the quimica-echem kernel."
    ) from exc

VeloxChem imported correctly: 1.0rc4
Python executable: c:\Users\Nayanika\miniconda3\envs\quimica-echem\python.exe


In [18]:
#@title Define your molecule using a SMILES code
import veloxchem as vlx
#@markdown Enter the SMILES code:
smiles_code = '[H]OC(=O)[C@@]([H])(N([H])[H])C([H])([H])c1c([H])c([H])c(N([H])[H])c([H])c1[H]' #@param {type:"string"}
molecule = vlx.Molecule.read_smiles(smiles_code)
print('Structure of the molecule entered:')
molecule.show(atom_indices=True)


Structure of the molecule entered:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [19]:
#@title DFT single point and charge calculation
#@markdown For classroom tests, start with a small basis such as STO-3G or DEF2-SVP.
basis_set = 'DEF2-SVP' #@param  ['STO-3G', 'DEF2-SV(P)', 'DEF2-SVP', 'DEF2-SVPD', 'DEF2-TZVP']
basis = vlx.MolecularBasis.read(molecule, basis_set)
scf_drv = vlx.ScfRestrictedDriver()
method_dict = {'conv_thresh': 1e-4}
scf_drv.update_settings(method_dict)
mute_output = True # @param {type:"boolean"}
if mute_output:
    scf_drv.ostream.mute()
functional = 'PBE' #@param ['SLATER', 'BLYP', 'B3LYP', 'PBE', 'PBE0', 'BP86']
scf_drv.xcfun = functional
scf_results = scf_drv.compute(molecule, basis)

esp_drv = vlx.EspChargesDriver()
esp_drv.ostream.mute()
esp_charges = esp_drv.compute(molecule, basis, scf_results)

resp_drv = vlx.RespChargesDriver()
resp_drv.ostream.mute()
resp_charges = resp_drv.compute(molecule, basis, scf_results)

chelpg_drv = vlx.EspChargesDriver()
chelpg_drv.ostream.mute()
chelpg_drv.grid_type = "chelpg"
chelpg_charges = chelpg_drv.compute(molecule, basis, scf_results)

for title, charges in [
    ("ESP charge", esp_charges),
    ("RESP charge", resp_charges),
    ("CHELPG charge", chelpg_charges),
]:
    print(f"Atom      {title}")
    print(20 * "-")
    for label, charge in zip(molecule.get_labels(), charges):
        print(f"{label :s} {charge : 18.6f}")
    print(20 * "-")
    print(f"Total: {charges.sum() : 13.6f}\n")


Atom      ESP charge
--------------------
O          -0.594956
C           0.487740
O          -0.466707
C           0.301119
N          -0.839100
C          -0.319573
C           0.108158
C          -0.148443
C          -0.274524
C           0.370512
N          -0.827155
C          -0.245209
C          -0.201948
H           0.438217
H           0.045723
H           0.339459
H           0.320714
H           0.083369
H           0.122217
H           0.143676
H           0.142915
H           0.363610
H           0.363856
H           0.137129
H           0.149203
--------------------
Total:      0.000000

Atom      RESP charge
--------------------
O          -0.571126
C           0.500492
O          -0.459312
C           0.150411
N          -0.798791
C          -0.099473
C           0.020193
C          -0.148640
C          -0.167618
C           0.185203
N          -0.709933
C          -0.131470
C          -0.200160
H           0.425031
H           0.069100
H           0.329334
H          

In [20]:
#@title Visualize the charges on molecule
import py3Dmol
viewer = py3Dmol.view(linked=False, width=500, height=500)

charge_type = 'ESP' # @param ["RESP", "ESP", "CHELPG"]
charge_map = {
    'ESP': esp_charges,
    'RESP': resp_charges,
    'CHELPG': chelpg_charges,
}
charges = charge_map[charge_type]
print(f'{charge_type} charges:')

# get_xyz_string() works for both SMILES-based and XYZ-based molecules.
# When using SMILES, VeloxChem generates 3D coordinates automatically via
# read_smiles(), so the geometry is always available here.
xyz_str = molecule.get_xyz_string()
viewer.addModel(xyz_str, "xyz")
for i, charge in enumerate(charges):
    viewer.addLabel(
        "{:.2f}".format(charge),
        {'backgroundOpacity': 0, 'fontColor': 'black', 'backgroundColor': 'white'},
        {'index': i},
    )
viewer.setStyle({"stick": {}, "sphere": {"radius": 0.4}})
viewer.rotate(-45, "y")
viewer.zoomTo()
viewer.show()

ESP charges:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.